In [2]:
import os
import SimpleITK as sitk

# Source paths
img_dir = r"E:\499\Data\task 1\ImagesTr"
lbl_dir = r"E:\499\Data\task 1\LabelsTr"

# Target nnUNet format
img_out = r"E:\nnUNet_raw\Dataset110_PANTHER_T1\imagesTr"
lbl_out = r"E:\nnUNet_raw\Dataset110_PANTHER_T1\labelsTr"

os.makedirs(img_out, exist_ok=True)
os.makedirs(lbl_out, exist_ok=True)

# Convert images
for f in os.listdir(img_dir):
    if f.endswith(".mha"):
        id = f.split("_")[0]
        img = sitk.ReadImage(os.path.join(img_dir, f))
        sitk.WriteImage(img, os.path.join(img_out, f"{id}_0000.nii.gz"))

# Convert labels
for f in os.listdir(lbl_dir):
    if f.endswith(".mha"):
        id = f.split("_")[0]
        lbl = sitk.ReadImage(os.path.join(lbl_dir, f))
        sitk.WriteImage(lbl, os.path.join(lbl_out, f"{id}.nii.gz"))

print("MHA to NIfTI conversion complete.")


✅ MHA to NIfTI conversion complete.


In [4]:
import os
import json

dataset_path = r"E:\nnUNet_raw\Dataset110_PANTHER_T1"
images_dir = os.path.join(dataset_path, "imagesTr")
labels_dir = os.path.join(dataset_path, "labelsTr")

image_files = sorted([f for f in os.listdir(images_dir) if f.endswith("_0000.nii.gz")])
label_files = sorted([f for f in os.listdir(labels_dir) if f.endswith(".nii.gz")])

training_pairs = []
for img_file in image_files:
    case_id = img_file.split("_")[0]
    lbl_file = f"{case_id}.nii.gz"
    if lbl_file in label_files:
        training_pairs.append({
            "image": f"./imagesTr/{img_file}",
            "label": f"./labelsTr/{lbl_file}"
        })

dataset_dict = {
    "name": "PANTHER_T1",
    "description": "Pancreatic tumor segmentation from T1 MRI",
    "tensorImageSize": "3D",
    "modality": {
        "0": "MRI"
    },
    "labels": {
        "0": "background",
        "1": "tumor",
        "2": "pancreas"
    },
    "numTraining": len(training_pairs),
    "numTest": 0,
    "training": training_pairs,
    "test": []
}

with open(os.path.join(dataset_path, "dataset.json"), "w") as f:
    json.dump(dataset_dict, f, indent=4)

print(f" dataset.json created with {len(training_pairs)} training samples.")


 dataset.json created with 92 training samples.


In [1]:
import os

os.environ['nnUNet_raw'] = 'E:/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = 'E:/nnUNet_preprocessed'
os.environ['nnUNet_results'] = 'E:/nnUNet_results'
os.environ["nnUNet_max_num_threads"] = "1"
os.environ["nnUNet_n_proc_DA"] = "1"




In [4]:
!nnUNetv2_extract_fingerprint -d 110


Dataset110_PANTHER_T1
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer



100%|##########| 92/92 [00:29<00:00,  3.10it/s]


In [12]:
!nnUNetv2_plan_and_preprocess -d 110


Fingerprint extraction...
Dataset110_PANTHER_T1
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [ 72. 258. 318.], 3d_lowres: [72, 258, 318]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 32, 'patch_size': (np.int64(320), np.int64(320)), 'median_image_size_in_voxels': array([258., 318.]), 'spacing': array([1.1875, 1.1875]), 'normalization_schemes': ['ZScoreNormalization'], 'use_mask_for_norm': [False], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resample_data_or_seg_to_shape', 'resampling_fn_data_kwargs': {'is_seg': False, 'order': 


100%|##########| 92/92 [02:05<00:00,  1.36s/it]

100%|##########| 92/92 [02:08<00:00,  1.39s/it]


In [ ]:
!nnUNetv2_train 110 3d_fullres 0 \
 -pretrained_weights "E:/499/Weights/checkpoint_best_PantherTask1_fold_0.pth"


In [ ]:
!nnUNetv2_train 110 3d_fullres 1 \
 -pretrained_weights "E:/499/Weights/checkpoint_best_PantherTask1_fold_1.pth"


In [ ]:
!nnUNetv2_train 110 3d_fullres 2 \
 -pretrained_weights "E:/499/Weights/checkpoint_best_PantherTask1_fold_2.pth"
